# Basic Linear Regression 

In [1]:
import numpy as np
import pyspark.sql.functions as F
import os

In [2]:
display(os.listdir('work'))

['housing_prices.csv', 'housing_prices_v1.ipynb', 'housing_prices_v2.ipynb']

### Step 1: Create Spark object

In [3]:
# No environment path hacking required!
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("HousingModelDocker") \
    .getOrCreate()

### Step 2: Create Dataframe

In [4]:
import os
display(os.listdir('work'))

['housing_prices.csv', 'housing_prices_v1.ipynb', 'housing_prices_v2.ipynb']

In [5]:
data = spark.read.csv('work/housing_prices.csv',header=True, inferSchema=True)
data.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



### Step 3: Data Cleansing

#### Counting NULL values in SINGLE column

In [6]:
(
    data.select
    (
        F.count(
                F.when(
                        F.col('ocean_proximity').isNull(),1
                    )
                ).alias('ocean_proximity')
    ).show()
 )

+---------------+
|ocean_proximity|
+---------------+
|              0|
+---------------+



#### Counting NULL values in all columns

In [7]:
(
    data.select
    (
        [
            F.count(
                F.when(
                        F.col(c).isNull(),1
                    )
                ).alias(c) for c in data.columns
        ]
    ).show()
)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|        0|       0|                 0|          0|           207|         0|         0|            0|                 0|              0|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+



#### Handling NULL value in ``total_bedrooms`` column

In [8]:
filtered_data = data.dropna(subset=['total_bedrooms'])
filtered_data.count()

20433

### Step 4: Split dataframe using ``randomSplit([0.8,0.2], seed=42)``

In [9]:
train_data, test_data = filtered_data.randomSplit([0.8, 0.2], seed=42)
print('Train size ', train_data.count())
print('Test size ', test_data.count())

Train size  16395
Test size  4038


### Step 5: Create Features using VectorAssembler

In [10]:
from pyspark.ml.feature import VectorAssembler

feature_columns = ['housing_median_age',
                   'total_rooms',
                   'total_bedrooms',
                   'population',
                   'households',
                   'median_income']
assemblers = VectorAssembler(inputCols=feature_columns,outputCol='features')

### Create training data

- use train_data from the split

In [11]:
transformed_trained_data = assemblers.transform(train_data)
transformed_trained_data.show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+--------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|            features|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+--------------------+
|  -124.35|   40.54|              52.0|     1820.0|         300.0|     806.0|     270.0|       3.0147|           94600.0|     NEAR OCEAN|[52.0,1820.0,300....|
|   -124.3|    41.8|              19.0|     2672.0|         552.0|    1298.0|     478.0|       1.9797|           85800.0|     NEAR OCEAN|[19.0,2672.0,552....|
|  -124.27|   40.69|              36.0|     2349.0|         528.0|    1194.0|     465.0|       2.5179|           79000.0|     NEAR OCEAN|[36.0,2349.0,528....|
|  -124.26|   40.58|              52.0|     22

### Create testing data

- use test_data from the split

In [12]:
transformed_test_data = assemblers.transform(test_data)
transformed_test_data.show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+--------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|            features|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+--------------------+
|   -124.3|   41.84|              17.0|     2677.0|         531.0|    1244.0|     456.0|       3.0313|          103600.0|     NEAR OCEAN|[17.0,2677.0,531....|
|  -124.23|   40.54|              52.0|     2694.0|         453.0|    1152.0|     435.0|       3.0806|          106700.0|     NEAR OCEAN|[52.0,2694.0,453....|
|  -124.23|   41.75|              11.0|     3159.0|         616.0|    1343.0|     479.0|       2.4805|           73200.0|     NEAR OCEAN|[11.0,3159.0,616....|
|  -124.19|   40.73|              21.0|     56

### Create Model

In [13]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol='features', labelCol='median_house_value', regParam=0.001)
model = lr.fit(transformed_trained_data)

predictions = model.transform(transformed_test_data)
predictions.show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+--------------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|            features|        prediction|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+--------------------+------------------+
|   -124.3|   41.84|              17.0|     2677.0|         531.0|    1244.0|     456.0|       3.0313|          103600.0|     NEAR OCEAN|[17.0,2677.0,531....| 143549.5515523904|
|  -124.23|   40.54|              52.0|     2694.0|         453.0|    1152.0|     435.0|       3.0806|          106700.0|     NEAR OCEAN|[52.0,2694.0,453....|204727.87978628205|
|  -124.23|   41.75|              11.0|     3159.0|         616.0|    1343.0|     479.0|       2.4805|        

### Evaluate using 

- Mean Absolute Error

Imagine you're guessing how many candies are in a big jar.

You guess 10 candies. There are actually 12 candies.

Your mistake is 2 candies too few.

Your friend guesses 15 candies. They are 5 candies too many.

Now, Mean Absolute Error (MAE) is like asking: "On average, how far off were our guesses?"

Step 1: Take the size of each mistake (ignore if too high or too low):

Your mistake: 2 candies

Friend's mistake: 5 candies

Step 2: Average them: (2 + 5) ÷ 2 = 3.5 candies

MAE = 3.5

That means: "Our guesses were wrong by 3.5 candies on average."

Lower MAE = Better guessing! 🎯

Real example: If your model predicts house prices, MAE = $20,000 means "On average, my predictions are off by $20,000."


In [14]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_mae = RegressionEvaluator(labelCol='median_house_value', predictionCol='prediction', metricName='mae')
mae = evaluator_mae.evaluate(predictions)
print(f'Mean Absolute Error (MAE): {mae}')

Mean Absolute Error (MAE): 56292.33482622761


# Model Evaluation

- Mean Absolute Error

Imagine you're guessing how many candies are in a big jar.

You guess 10 candies. There are actually 12 candies.

Your mistake is 2 candies too few.

Your friend guesses 15 candies. They are 5 candies too many.

Now, Mean Absolute Error (MAE) is like asking: "On average, how far off were our guesses?"

Step 1: Take the size of each mistake (ignore if too high or too low):

Your mistake: 2 candies

Friend's mistake: 5 candies

Step 2: Average them: (2 + 5) ÷ 2 = 3.5 candies

MAE = 3.5

That means: "Our guesses were wrong by 3.5 candies on average."

Lower MAE = Better guessing! 🎯

Real example: If your model predicts house prices, MAE = $20,000 means "On average, my predictions are off by $20,000."

- Root Mean Squared Error

You guess: 10 candies (actual: 12) → Mistake = 2
Friend guesses: 15 candies (actual: 12) → Mistake = 3

MAE (like before):
Just average: (2 + 3) ÷ 2 = 2.5 candies ❌

RMSE (punishes big mistakes):
Step 1: Square the mistakes (big ones hurt more!)

Your mistake: 2² = 4

Friend's mistake: 3² = 9
Step 2: Average: (4 + 9) ÷ 2 = 6.5
Step 3: Square root: √6.5 ≈ 2.55 candies